In [ ]:
import os
import torch
import logging
import numpy as np
import netCDF4 as nc
#from functools import partial
#from typing import Any, Callable, Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import xarray as xr
import dask
import cartopy.crs as ccrs

In [ ]:
torch.cuda.device_count()

# Load dataset

In [ ]:
import time
import numpy as np
import xarray as xr
import dask
from concurrent.futures import ThreadPoolExecutor  # ✅ safer for xarray+Dask

# Enable Dask threading
dask.config.set(scheduler='threads')

start_time = time.time()

variable_names = ['LWCF', 'SWCF', 'FLUT', 'FSNT', 'TGCLDLWP']

def load_member_all_vars(member, scheme):
    var_datasets = {}
    for var in variable_names:
        file_path = f'/glade/derecho/scratch/fanda/FROM_CHEYENNE/PPE_monthly/{scheme}/PPE_100_{scheme}.{member:02}_daily_{var}.nc'
        ds = xr.open_dataset(file_path, chunks={'time': 1}, decode_times=True, use_cftime=True)  # ✅ safe time handling
        var_datasets[var] = ds[var]
    combined_ds = xr.Dataset(var_datasets)
    combined_ds = combined_ds.expand_dims(member=[member])
    return combined_ds

def load_ensemble_data(scheme, num_members):
    num_workers = min(8, num_members)
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        ensemble_members = list(executor.map(load_member_all_vars, range(1, num_members + 1), [scheme] * num_members))
    return xr.concat(ensemble_members, dim='member')

ensemble_KK_daily = load_ensemble_data('kk', 100)
ensemble_ML_daily = load_ensemble_data('ml', 100)

end_time = time.time()
print(f"Execution time: {end_time - start_time:.2f} seconds")

In [ ]:
ensemble_KK_daily

In [ ]:
interpolated_monthly_clim_obs.isel(obstype=1)

In [ ]:
# normalization
import cftime

# Step 1: Remove the first month (0001-01) and keep until 0003-12
start = cftime.DatetimeNoLeap(1, 2, 1)
end = cftime.DatetimeNoLeap(3, 12, 31)

# Use boolean indexing instead of slice
ML_monthly_data = ensemble_ML_daily.where(
    (ensemble_ML_daily.time >= start) & (ensemble_ML_daily.time <= end),
    drop=True
)
KK_monthly_data = ensemble_KK_daily.where(
    (ensemble_KK_daily.time >= start) & (ensemble_KK_daily.time <= end),
    drop=True
)


# Step 2: Assign 'month' coordinate from cftime time values
ML_monthly_data = ML_monthly_data.assign_coords(
    month=("time", [f"{t.year:04d}-{t.month:02d}" for t in ML_monthly_data['time'].values])
)
KK_monthly_data = KK_monthly_data.assign_coords(
    month=("time", [f"{t.year:04d}-{t.month:02d}" for t in KK_monthly_data['time'].values])
)

# Step 3: Compute monthly means
ML_monthly_mean = ML_monthly_data.groupby('month').mean('time')
KK_monthly_mean = KK_monthly_data.groupby('month').mean('time')

# Step 4: Split into train/val/test based on member slices
train_members = slice(1, 60)
val_members = slice(61, 80)
test_members = slice(81, 100)

ML_train = ML_monthly_mean.sel(member=train_members)
KK_train = KK_monthly_mean.sel(member=train_members)

ML_val = ML_monthly_mean.sel(member=val_members)
KK_val = KK_monthly_mean.sel(member=val_members)

ML_test = ML_monthly_mean.sel(member=test_members)
KK_test = KK_monthly_mean.sel(member=test_members)

# Step 5: Normalize each dataset using training mean/std over 'month'
ML_train_normalized = (ML_train - ML_train.mean(dim=("month","lat", "lon"))) / ML_train.std(dim=("month","lat", "lon"))
KK_train_normalized = (KK_train - KK_train.mean(dim=("month","lat", "lon"))) / KK_train.std(dim=("month","lat", "lon"))

ML_val_normalized = (ML_val - ML_val.mean(dim=("month","lat", "lon"))) / ML_val.std(dim=("month","lat", "lon"))
KK_val_normalized = (KK_val - KK_val.mean(dim=("month","lat", "lon"))) / KK_val.std(dim=("month","lat", "lon"))

ML_test_normalized = (ML_test - ML_test.mean(dim=("month","lat", "lon"))) / ML_test.std(dim=("month","lat", "lon"))
KK_test_normalized = (KK_test - KK_test.mean(dim=("month","lat", "lon"))) / KK_test.std(dim=("month","lat", "lon"))

# Step 6: Handle NaNs
ML_train_normalized = ML_train_normalized.fillna(0)
KK_train_normalized = KK_train_normalized.fillna(0)

ML_val_normalized = ML_val_normalized.fillna(0)
KK_val_normalized = KK_val_normalized.fillna(0)

ML_test_normalized = ML_test_normalized.fillna(0)
KK_test_normalized = KK_test_normalized.fillna(0)

# Step 7: Print shapes
'''
print("Training ML Normalized Shape:", ML_train_normalized.shape)
print("Validation ML Normalized Shape:", ML_val_normalized.shape)
print("Testing ML Normalized Shape:", ML_test_normalized.shape)
'''

In [ ]:
import random
import torch
import xarray as xr
import dask.array as da
from torch.utils.data import Dataset
from concurrent.futures import ThreadPoolExecutor
class MemberPairDataset(Dataset):
    def __init__(self, ml_data=None, kk_data=None, num_workers=8, random_seed=42, preloaded_data=None):
        self.num_workers = num_workers

        if preloaded_data:
            self.data = preloaded_data["data"]
            self.labels = preloaded_data["labels"]
        else:
            if ml_data is None or kk_data is None:
                raise ValueError("ml_data and kk_data must be provided if not loading from file")

            self.ml_data = ml_data  # xarray.Dataset: (month, member, lat, lon) with many vars
            self.kk_data = kk_data

            random.seed(random_seed)

            self.ml_pairs, self.ml_meta = self._generate_member_pairs(self.ml_data)
            self.kk_pairs, self.kk_meta = self._generate_member_pairs(self.kk_data)

            self.data = self.ml_pairs + self.kk_pairs
            self.labels = [0] * len(self.ml_pairs) + [1] * len(self.kk_pairs)
            self.meta = self.ml_meta + self.kk_meta

            self.members = xr.DataArray([m for m, _ in self.meta], dims="pairs")
            self.months = xr.DataArray([mo for _, mo in self.meta], dims="pairs")

    def _generate_member_pairs(self, data):
        num_members = data.sizes["member"]
        num_time_steps = data.sizes["month"]

        pairs = []
        meta = []

        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            futures = []
            for m in range(num_members):
                for t in range(num_time_steps - 1):
                    futures.append(executor.submit(self._create_pair, data, m, t))

            for future in futures:
                pair, info = future.result()
                pairs.append(pair)
                meta.append(info)

        return pairs, meta

    def _create_pair(self, data, member_idx, month_idx):
        # Direct index access using .isel
        slice1 = data.isel(month=month_idx, member=member_idx)
        slice2 = data.isel(month=month_idx + 1, member=member_idx)
        # Stack each variable across a new 'var' dim
        vstack1 = xr.concat([slice1[v] for v in data.data_vars], dim="var")
        vstack2 = xr.concat([slice2[v] for v in data.data_vars], dim="var")

        # Combine into pair along a 'pair' dimension
        pair = xr.concat([vstack1, vstack2], dim="pair")

        # Still record the actual month string (optional metadata)
        m1_label = str(data["month"].values[month_idx])
        return pair, (member_idx, m1_label)


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx].load().values, dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

    def save_to_zarr(self, filepath):
        data = xr.concat(self.data, dim="pairs")
        data = data.assign_coords(pairs=("pairs", range(len(self.data))))
        labels = xr.DataArray(self.labels, dims="pairs", name="labels")

        dataset = xr.Dataset({
            "data": data.chunk({"pairs": 1}),
            "labels": labels
        })
        dataset.to_zarr(filepath, mode="w", consolidated=True)

    @classmethod
    def load_from_zarr(cls, filepath):
        ds = xr.open_zarr(filepath)
        return cls(preloaded_data={"data": ds["data"], "labels": ds["labels"]})


In [ ]:
# Save the weekly file

#ML_train_normalized.sel(lat=slice(-65.5, 65.5))
lat_threshold = 65.5
# Start timing
start_time = time.time()

# Data Preparation
# Generate training pairs
train_dataset = MemberPairDataset(ML_train_normalized.sel(lat=slice(-lat_threshold, lat_threshold)), KK_train_normalized.sel(lat=slice(-lat_threshold, lat_threshold)))

# Generate validation pairs
val_dataset = MemberPairDataset(ML_val_normalized.sel(lat=slice(-lat_threshold, lat_threshold)), KK_val_normalized.sel(lat=slice(-lat_threshold, lat_threshold)))

# Generate testing pairs
test_dataset = MemberPairDataset(ML_test_normalized.sel(lat=slice(-lat_threshold, lat_threshold)), KK_test_normalized.sel(lat=slice(-lat_threshold, lat_threshold)))

# Example: Retrieve a data sample
#x1, x2, label = train_dataset[0]
#print(f"Pair shapes: {x1.shape}, {x2.shape}, Label: {label}")

# End timing
end_time = time.time()

# Print execution time
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.2f} seconds")

# Save the dataset in Zarr format
# Start timing
start_time = time.time()
train_dataset.save_to_zarr("../data/train_dataset_monthly_radiation_LWP_no_poles.zarr")
print("Training dataset saved in Zarr format!")
# End timing
end_time = time.time()

# Print execution time
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.2f} seconds")

val_dataset.save_to_zarr("../data/val_dataset_monthly_radiation_LWP_no_poles.zarr")
print("Validation dataset saved in Zarr format!")

test_dataset.save_to_zarr("../data/test_dataset_monthly_radiation_LWP_no_poles.zarr")
print("Testing dataset saved in Zarr format!")

In [ ]:
import xarray as xr

# Load dataset from Zarr
train_dataset = xr.open_zarr("../data/train_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
val_dataset = xr.open_zarr("../data/val_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
test_dataset  = xr.open_zarr("../data/test_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)

In [ ]:
train_dataset